# บทที่ 9: Graph (กราฟ) และการท่องกราฟ

> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)
> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*
> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)

---

## 9.1 การแทนกราฟ

กราฟ \(G = (V, E)\) มีจุด (vertex) และเส้นเชื่อม (edge) อาจเป็น *directed/undirected*, *weighted/unweighted*

| การแทน | หน่วยความจำ | ตรวจว่ามี edge (u,v) | ไล่เพื่อนบ้านของ u |
|---|---|---|---|
| Adjacency **matrix** | \(O(V^2)\) | \(O(1)\) | \(O(V)\) |
| Adjacency **list** | \(O(V+E)\) | \(O(\deg u)\) | \(O(\deg u)\) |

กราฟจริงส่วนใหญ่ *sparse* → นิยมใช้ adjacency list

In [ ]:
from collections import defaultdict, deque

class Graph:
    def __init__(self, directed=False):
        self.adj = defaultdict(list)
        self.directed = directed

    def add_edge(self, u, v, w=1):
        self.adj[u].append((v, w))
        if not self.directed:
            self.adj[v].append((u, w))
        else:
            self.adj.setdefault(v, [])

    def vertices(self):
        return list(self.adj.keys())

    def matrix(self):
        # แปลงเป็น adjacency matrix เพื่อเปรียบเทียบสองรูปแบบ
        vs = sorted(self.adj)
        idx = {v: i for i, v in enumerate(vs)}
        M = [[0] * len(vs) for _ in vs]
        for u in vs:
            for v, w in self.adj[u]:
                M[idx[u]][idx[v]] = w
        return vs, M

g = Graph()
for u, v in [('A','B'), ('A','C'), ('B','D'), ('C','D'), ('D','E')]:
    g.add_edge(u, v)
print("adjacency list:")
for u in sorted(g.adj):
    print(f"  {u}: {[v for v, _ in g.adj[u]]}")
vs, M = g.matrix()
print("adjacency matrix:", vs)
for row in M: print("  ", row)

## 9.2 BFS และ DFS

| | BFS | DFS |
|---|---|---|
| โครงสร้างช่วย | Queue | Stack / recursion |
| ลำดับสำรวจ | ทีละชั้นจากจุดเริ่ม | ลงลึกให้สุดก่อนถอยกลับ |
| Complexity | \(O(V+E)\) | \(O(V+E)\) |
| ใช้ทำ | เส้นทางสั้นสุดบนกราฟไร้น้ำหนัก, หาระดับชั้น | ตรวจ cycle, topological sort, หา connected component |

In [ ]:
from collections import deque

def bfs(g, start):
    visited, order, q = {start}, [], deque([start])
    while q:
        u = q.popleft()
        order.append(u)
        for v, _ in g.adj[u]:
            if v not in visited:
                visited.add(v)          # mark ตอน enqueue กัน node เข้าคิวซ้ำ
                q.append(v)
    return order

def bfs_shortest_path(g, start, goal):
    # กราฟไร้น้ำหนัก: BFS ให้เส้นทางที่สั้นสุด (นับจำนวน edge)
    prev, q = {start: None}, deque([start])
    while q:
        u = q.popleft()
        if u == goal:
            path = []
            while u is not None:
                path.append(u); u = prev[u]
            return path[::-1]
        for v, _ in g.adj[u]:
            if v not in prev:
                prev[v] = u; q.append(v)
    return None

def dfs_recursive(g, u, visited=None, order=None):
    visited = set() if visited is None else visited
    order = [] if order is None else order
    visited.add(u); order.append(u)
    for v, _ in g.adj[u]:
        if v not in visited:
            dfs_recursive(g, v, visited, order)
    return order

def dfs_iterative(g, start):
    visited, order, st = set(), [], [start]
    while st:
        u = st.pop()
        if u in visited: continue
        visited.add(u); order.append(u)
        for v, _ in reversed(g.adj[u]):
            if v not in visited: st.append(v)
    return order

print("BFS      :", bfs(g, 'A'))
print("DFS (rec):", dfs_recursive(g, 'A'))
print("DFS (it) :", dfs_iterative(g, 'A'))
print("path A->E:", bfs_shortest_path(g, 'A', 'E'))

In [ ]:
# Topological sort (เฉพาะ DAG) 2 วิธี: Kahn (BFS) และ DFS
from collections import defaultdict, deque

def topo_kahn(g):
    indeg = {u: 0 for u in g.adj}
    for u in g.adj:
        for v, _ in g.adj[u]:
            indeg[v] += 1
    q = deque([u for u in indeg if indeg[u] == 0])
    order = []
    while q:
        u = q.popleft(); order.append(u)
        for v, _ in g.adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0: q.append(v)
    if len(order) != len(indeg):
        raise ValueError('กราฟมี cycle -> ไม่สามารถ topological sort ได้')
    return order

def has_cycle_directed(g):
    WHITE, GRAY, BLACK = 0, 1, 2
    color = {u: WHITE for u in g.adj}
    def dfs(u):
        color[u] = GRAY                       # GRAY = อยู่ใน recursion stack
        for v, _ in g.adj[u]:
            if color[v] == GRAY: return True  # เจอ back edge -> มี cycle
            if color[v] == WHITE and dfs(v): return True
        color[u] = BLACK
        return False
    return any(color[u] == WHITE and dfs(u) for u in list(g.adj))

# ตัวอย่าง: ลำดับวิชาที่ต้องเรียนก่อน (prerequisite)
courses = Graph(directed=True)
for a, b in [('Python','DS'), ('Math','DS'), ('DS','Algo'), ('DS','DB'), ('Algo','AI')]:
    courses.add_edge(a, b)
print("ลำดับเรียน:", topo_kahn(courses))
print("มี cycle?  :", has_cycle_directed(courses))

In [ ]:
# Dijkstra: เส้นทางสั้นสุดบนกราฟถ่วงน้ำหนัก (น้ำหนักต้องไม่ติดลบ) -> O((V+E) log V)
import heapq

def dijkstra(g, src):
    dist = {u: float('inf') for u in g.adj}
    prev = {u: None for u in g.adj}
    dist[src] = 0
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue          # รายการเก่าที่ล้าสมัย -> ข้าม
        for v, w in g.adj[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v], prev[v] = nd, u
                heapq.heappush(pq, (nd, v))
    return dist, prev

def rebuild_path(prev, goal):
    path = []
    while goal is not None:
        path.append(goal); goal = prev[goal]
    return path[::-1]

wg = Graph()
for u, v, w in [('A','B',4), ('A','C',2), ('B','C',5), ('B','D',10),
                ('C','E',3), ('E','D',4), ('D','F',11)]:
    wg.add_edge(u, v, w)
dist, prev = dijkstra(wg, 'A')
print("ระยะทางจาก A:", {k: v for k, v in sorted(dist.items())})
print("เส้นทาง A->F :", rebuild_path(prev, 'F'), "ระยะ =", dist['F'])

In [ ]:
# Union-Find (Disjoint Set) + Kruskal MST
class UnionFind:
    def __init__(self, items):
        self.p = {x: x for x in items}
        self.r = {x: 0 for x in items}
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]        # path compression
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return False                # อยู่กลุ่มเดียวกันแล้ว -> จะเกิด cycle
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]: self.r[ra] += 1
        return True

def kruskal(vertices, edges):
    # edges = [(w, u, v)] ; เรียงน้ำหนักน้อยไปมาก แล้วเลือกที่ไม่ทำให้เกิด cycle
    uf = UnionFind(vertices)
    mst, total = [], 0
    for w, u, v in sorted(edges):
        if uf.union(u, v):
            mst.append((u, v, w)); total += w
    return mst, total

V = list("ABCDEF")
E = [(4,'A','B'), (2,'A','C'), (5,'B','C'), (10,'B','D'),
     (3,'C','E'), (4,'E','D'), (11,'D','F')]
mst, total = kruskal(V, E)
print("MST:", mst)
print("น้ำหนักรวม =", total)

---
## แบบฝึกหัดบทที่ 9

1. นับจำนวน connected component ของกราฟไร้ทิศทาง
2. ตรวจว่ากราฟเป็น bipartite หรือไม่ (ระบายสี 2 สีด้วย BFS)
3. นับจำนวน "เกาะ" ในตารางกริด 0/1 (grid เป็นกราฟโดยปริยาย)
4. หาเส้นทางสั้นสุดในเขาวงกต (maze) จากมุมซ้ายบนไปขวาล่าง

### เฉลยบทที่ 9

In [ ]:
from collections import deque

# ข้อ 1
def count_components(g):
    seen, c = set(), 0
    for u in g.adj:
        if u not in seen:
            c += 1
            for v in bfs(g, u): seen.add(v)
    return c

# ข้อ 2: ระบายสีสลับ ถ้าเพื่อนบ้านสีเดียวกัน -> ไม่ bipartite
def is_bipartite(g):
    color = {}
    for s in g.adj:
        if s in color: continue
        color[s] = 0; q = deque([s])
        while q:
            u = q.popleft()
            for v, _ in g.adj[u]:
                if v not in color:
                    color[v] = 1 - color[u]; q.append(v)
                elif color[v] == color[u]:
                    return False
    return True

# ข้อ 3: DFS/flood fill บนกริด -> O(R*C)
def count_islands(grid):
    if not grid: return 0
    R, C = len(grid), len(grid[0])
    seen = set()
    def flood(r, c):
        st = [(r, c)]
        while st:
            i, j = st.pop()
            for di, dj in ((1,0), (-1,0), (0,1), (0,-1)):
                ni, nj = i+di, j+dj
                if 0 <= ni < R and 0 <= nj < C and grid[ni][nj] == 1 and (ni,nj) not in seen:
                    seen.add((ni,nj)); st.append((ni,nj))
    n = 0
    for r in range(R):
        for c in range(C):
            if grid[r][c] == 1 and (r,c) not in seen:
                n += 1; seen.add((r,c)); flood(r, c)
    return n

# ข้อ 4: BFS บนกริด (0 = เดินได้, 1 = กำแพง)
def maze_shortest(maze):
    R, C = len(maze), len(maze[0])
    q, dist = deque([(0,0)]), {(0,0): 1}
    while q:
        r, c = q.popleft()
        if (r, c) == (R-1, C-1): return dist[(r,c)]
        for dr, dc in ((1,0), (-1,0), (0,1), (0,-1)):
            nr, nc = r+dr, c+dc
            if 0 <= nr < R and 0 <= nc < C and maze[nr][nc] == 0 and (nr,nc) not in dist:
                dist[(nr,nc)] = dist[(r,c)] + 1
                q.append((nr,nc))
    return -1

g2 = Graph()
g2.add_edge('A','B'); g2.add_edge('C','D'); g2.add_edge('E','E')
print("components =", count_components(g2))
print("bipartite? =", is_bipartite(g))
print("islands =", count_islands([[1,1,0,0],[1,0,0,1],[0,0,1,1]]))
print("maze steps =", maze_shortest([[0,0,1],[1,0,1],[1,0,0]]))